# MarginGraph: LLM rerank (E5/E6) on a free GPU

Run on Kaggle or Colab with a T4. Upload `artifacts/test_slates.parquet` and `artifacts/catalog.parquet` from the CPU run. The LLM scores the same top-20 candidates that LightGBM ranked, so the comparison is paired.

Go/no-go: the LLM (or the blend) must beat LightGBM with a paired CI above zero.

In [ ]:
!pip install -q git+https://github.com/Sanskar1724/MarginGraph transformers accelerate
# No tokens needed: Qwen2.5 is ungated. Never paste keys into the notebook; use Kaggle/Colab secrets.

In [ ]:
import pandas as pd
from margingraph.llm import score_slates, compare
slates = pd.read_parquet('test_slates.parquet')
catalog = pd.read_parquet('catalog.parquet')
# start small to check timing, then raise
ids = slates.event_id.drop_duplicates().sample(1000, random_state=0)
sub = slates[slates.event_id.isin(ids)].copy()
len(sub)

In [ ]:
sub['llm'] = score_slates(sub, catalog, 'Qwen/Qwen2.5-1.5B-Instruct', batch_size=32)
sub.to_parquet('llm_scores_qwen1.5b.parquet')
compare(sub).round(4)

If the 1.5B model is close, try `Qwen/Qwen2.5-3B-Instruct`, then fine-tune (E6) with hard negatives from the LightGBM slates.